In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# =========================================================
# USER INPUTS
# =========================================================
OUTCOME_FILE = "Outcome_Master_Modeling_Panel_v1.csv"
OUTCOME_COL = "outcome_dissident_favorable"   # positive class = dissident-favorable outcome
OUTPUT_DIR = "."  # change if you want a different folder

# =========================================================
# HELPERS
# =========================================================
def resolve_model_year(df: pd.DataFrame) -> pd.Series:
    """
    Resolve the campaign year used for chronological splitting.
    Priority:
    1) announcement_date-like columns
    2) explicit year columns
    """
    date_candidates = [
        "announcement_date",
        "campaign_announcement_date",
        "announced_date",
        "launch_date",
        "date"
    ]
    year_candidates = [
        "year",
        "campaign_year",
        "announcement_year",
        "launch_year"
    ]

    for col in date_candidates:
        if col in df.columns:
            return pd.to_datetime(df[col], errors="coerce").dt.year

    for col in year_candidates:
        if col in df.columns:
            return pd.to_numeric(df[col], errors="coerce").astype("Int64")

    raise ValueError(
        "Could not find a usable campaign year column. "
        "Please add one manually or update resolve_model_year()."
    )


def safe_roc_auc(y_true: pd.Series, y_score: np.ndarray) -> float:
    """
    ROC-AUC is undefined if only one class is present.
    """
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


def evaluate_constant_baseline(y_true: pd.Series, constant_prob: float) -> dict:
    """
    Evaluate a constant-probability baseline on a target vector.
    """
    y_true = pd.Series(y_true).astype(int)
    y_pred = np.full(len(y_true), float(constant_prob))

    return {
        "n_obs": int(len(y_true)),
        "positive_rate": float(y_true.mean()),
        "predicted_prob": float(constant_prob),
        "pr_auc": float(average_precision_score(y_true, y_pred)),
        "roc_auc": float(safe_roc_auc(y_true, y_pred)),
        "brier": float(brier_score_loss(y_true, y_pred))
    }


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(OUTCOME_FILE).copy()

# Keep only rows with nonmissing outcome
df = df.loc[df[OUTCOME_COL].notna()].copy()
df[OUTCOME_COL] = pd.to_numeric(df[OUTCOME_COL], errors="coerce").astype(int)

# Resolve year for chronological splitting
df["model_year"] = resolve_model_year(df)

# Drop rows where year could not be resolved
df = df.loc[df["model_year"].notna()].copy()
df["model_year"] = df["model_year"].astype(int)

print("Resolved sample rows:", len(df))
print("Overall positive rate:", round(df[OUTCOME_COL].mean(), 6))
print("Years covered:", sorted(df["model_year"].unique()))

# =========================================================
# FULL-SAMPLE EXTENSION BASELINE
# Same chronological structure as the main extension horse race:
# CV folds:
#   2010-2015 -> 2016
#   2010-2016 -> 2017
#   2010-2017 -> 2018
#   2010-2018 -> 2019
#   2010-2019 -> 2020
#   2010-2020 -> 2021
# Final holdout:
#   2022-2024
# =========================================================
fold_rows = []

fold_train_end_years = [2015, 2016, 2017, 2018, 2019, 2020]
fold_validation_years = [2016, 2017, 2018, 2019, 2020, 2021]

for train_end, val_year in zip(fold_train_end_years, fold_validation_years):
    train_mask = df["model_year"] <= train_end
    val_mask = df["model_year"] == val_year

    train = df.loc[train_mask].copy()
    val = df.loc[val_mask].copy()

    train_prevalence = train[OUTCOME_COL].mean()
    metrics = evaluate_constant_baseline(val[OUTCOME_COL], train_prevalence)

    fold_rows.append({
        "fold": f"{int(train['model_year'].min())}-{train_end} -> {val_year}",
        "train_end_year": train_end,
        "validation_year": val_year,
        "train_n": int(len(train)),
        "validation_n": int(len(val)),
        "train_prevalence": float(train_prevalence),
        **metrics
    })

cv_df = pd.DataFrame(fold_rows)

# CV summary
cv_summary = pd.DataFrame([{
    "mean_cv_pr_auc": cv_df["pr_auc"].mean(),
    "mean_cv_roc_auc": cv_df["roc_auc"].mean(),
    "mean_cv_brier": cv_df["brier"].mean()
}])

# Final refit / holdout
refit_mask = df["model_year"] <= 2021
holdout_mask = df["model_year"].isin([2022, 2023, 2024])

refit = df.loc[refit_mask].copy()
holdout = df.loc[holdout_mask].copy()

refit_prevalence = refit[OUTCOME_COL].mean()
holdout_metrics = evaluate_constant_baseline(holdout[OUTCOME_COL], refit_prevalence)

holdout_df = pd.DataFrame([{
    "refit_years": f"{int(refit['model_year'].min())}-2021",
    "holdout_years": "2022-2024",
    "refit_n": int(len(refit)),
    "holdout_n": int(len(holdout)),
    "refit_prevalence": float(refit_prevalence),
    **holdout_metrics
}])

# =========================================================
# SAVE OUTPUTS
# =========================================================
fold_path = f"{OUTPUT_DIR}/outcome_baseline_fold_results.csv"
cv_summary_path = f"{OUTPUT_DIR}/outcome_baseline_cv_summary.csv"
holdout_path = f"{OUTPUT_DIR}/outcome_baseline_holdout_summary.csv"

cv_df.to_csv(fold_path, index=False)
cv_summary.to_csv(cv_summary_path, index=False)
holdout_df.to_csv(holdout_path, index=False)

# =========================================================
# PRINT RESULTS
# =========================================================
print("\n=== FULL-SAMPLE OUTCOME EXTENSION BASELINE: FOLD RESULTS ===")
print(cv_df.round(6).to_string(index=False))

print("\n=== FULL-SAMPLE OUTCOME EXTENSION BASELINE: CV SUMMARY ===")
print(cv_summary.round(6).to_string(index=False))

print("\n=== FULL-SAMPLE OUTCOME EXTENSION BASELINE: HOLDOUT ===")
print(holdout_df.round(6).to_string(index=False))

print("\nSaved files:")
print(fold_path)
print(cv_summary_path)
print(holdout_path)

Resolved sample rows: 1401
Overall positive rate: 0.310493
Years covered: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

=== FULL-SAMPLE OUTCOME EXTENSION BASELINE: FOLD RESULTS ===
             fold  train_end_year  validation_year  train_n  validation_n  train_prevalence  n_obs  positive_rate  predicted_prob   pr_auc  roc_auc    brier
2010-2015 -> 2016            2015             2016      682            91          0.401760     91       0.186813        0.401760 0.186813      0.5 0.198116
2010-2016 -> 2017            2016             2017      773            70          0.376455     70       0.114286        0.376455 0.114286      0.5 0.169957
2010-2017 -> 2018            2017             2018      843           100          0.354686    100       0.230000        0.354686 0.230000      0.5 0.192647
2010-2018 -> 2019            2018             2019      943            83          0.341463     83       0.192771        0.341463 0.192771      0